In [1]:

import os
import random
import numpy as np
import pandas as pd
import tensorflow as tf
from pathlib import Path

# Smart Crop Care configuration
SEED = 42
IMG_SIZE = (224, 224)
BATCH_SIZE = 32
TRAIN_FRACTION = 0.50

# Dataset paths
DATA_ROOT = Path(
    "/kaggle/input/datasets/vipoooool/"
    "new-plant-diseases-dataset/"
    "New Plant Diseases Dataset(Augmented)/"
    "New Plant Diseases Dataset(Augmented)"
)

TRAIN_DIR = DATA_ROOT / "train"
VALID_DIR = DATA_ROOT / "valid"

IMAGE_EXTENSIONS = {
    ".jpg", ".jpeg", ".png", ".bmp", ".webp"
}

# Verify dataset folders
print("TensorFlow version:", tf.__version__)
print("Training folder exists:", TRAIN_DIR.is_dir())
print("Validation folder exists:", VALID_DIR.is_dir())
print("GPU devices:", tf.config.list_physical_devices("GPU"))

assert TRAIN_DIR.is_dir(), "Training folder not found!"
assert VALID_DIR.is_dir(), "Validation folder not found!"

class_names = sorted(
    p.name for p in TRAIN_DIR.iterdir()
    if p.is_dir()
)

NUM_CLASSES = len(class_names)

print("Number of classes:", NUM_CLASSES)
print("Image resolution:", IMG_SIZE)
print("Batch size:", BATCH_SIZE)
print("Training fraction:", TRAIN_FRACTION)

assert NUM_CLASSES == 38, "Expected 38 classes."

print("\nSTEP 1 COMPLETED SUCCESSFULLY")


TensorFlow version: 2.20.0
Training folder exists: True
Validation folder exists: True
GPU devices: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU'), PhysicalDevice(name='/physical_device:GPU:1', device_type='GPU')]
Number of classes: 38
Image resolution: (224, 224)
Batch size: 32
Training fraction: 0.5

STEP 1 COMPLETED SUCCESSFULLY


In [2]:

# STEP 2: Verify dataset classes and image counts

train_counts = {}
valid_counts = {}

for class_name in class_names:
    train_class_dir = TRAIN_DIR / class_name
    valid_class_dir = VALID_DIR / class_name

    train_images = [
        p for p in train_class_dir.iterdir()
        if p.is_file()
        and p.suffix.lower() in IMAGE_EXTENSIONS
    ]

    valid_images = [
        p for p in valid_class_dir.iterdir()
        if p.is_file()
        and p.suffix.lower() in IMAGE_EXTENSIONS
    ]

    train_counts[class_name] = len(train_images)
    valid_counts[class_name] = len(valid_images)

    assert len(train_images) > 0, (
        f"No training images in {class_name}"
    )
    assert len(valid_images) > 0, (
        f"No validation images in {class_name}"
    )

print("Number of classes:", len(class_names))
print("Total original training images:", sum(train_counts.values()))
print("Total validation images:", sum(valid_counts.values()))
print("Classes with training images:", sum(
    count > 0 for count in train_counts.values()
))
print("Classes with validation images:", sum(
    count > 0 for count in valid_counts.values()
))

print("\nFirst 5 classes:")
for name in class_names[:5]:
    print(
        f"{name}: "
        f"train={train_counts[name]}, "
        f"validation={valid_counts[name]}"
    )

print("\nSTEP 2 COMPLETED SUCCESSFULLY")


Number of classes: 38
Total original training images: 70295
Total validation images: 17572
Classes with training images: 38
Classes with validation images: 38

First 5 classes:
Apple___Apple_scab: train=2016, validation=504
Apple___Black_rot: train=1987, validation=497
Apple___Cedar_apple_rust: train=1760, validation=440
Apple___healthy: train=2008, validation=502
Blueberry___healthy: train=1816, validation=454

STEP 2 COMPLETED SUCCESSFULLY


In [3]:

# STEP 3: Create reproducible training, validation, and test splits

import random
import pandas as pd

SEED = 42
random.seed(SEED)

train_records = []
test_records = []
valid_records = []

for class_name in class_names:
    # Get original training images for this class
    image_paths = [
        p for p in (TRAIN_DIR / class_name).iterdir()
        if p.is_file() and p.suffix.lower() in IMAGE_EXTENSIONS
    ]

    # Shuffle reproducibly
    image_paths = sorted(image_paths)
    random.Random(f"{SEED}_{class_name}").shuffle(image_paths)

    total = len(image_paths)

    # Reserve 10% for testing
    test_count = max(1, round(total * 0.10))

    # Select 50% of the original images for training
    train_count = round(total * 0.50)

    test_paths = image_paths[:test_count]
    train_paths = image_paths[test_count:test_count + train_count]

    for path in train_paths:
        train_records.append({
            "image_path": str(path),
            "label": class_name
        })

    for path in test_paths:
        test_records.append({
            "image_path": str(path),
            "label": class_name
        })

    # Keep the existing validation dataset unchanged
    valid_paths = [
        p for p in (VALID_DIR / class_name).iterdir()
        if p.is_file() and p.suffix.lower() in IMAGE_EXTENSIONS
    ]

    for path in valid_paths:
        valid_records.append({
            "image_path": str(path),
            "label": class_name
        })

train_df = pd.DataFrame(train_records)
test_df = pd.DataFrame(test_records)
valid_df = pd.DataFrame(valid_records)

# Verify that training and test images do not overlap
train_paths_set = set(train_df["image_path"])
test_paths_set = set(test_df["image_path"])

assert train_paths_set.isdisjoint(test_paths_set)
assert train_df["label"].nunique() == len(class_names)
assert test_df["label"].nunique() == len(class_names)
assert valid_df["label"].nunique() == len(class_names)

print("Training images:", len(train_df))
print("Validation images:", len(valid_df))
print("Test images:", len(test_df))

print("\nClasses in each split:")
print("Train:", train_df["label"].nunique())
print("Validation:", valid_df["label"].nunique())
print("Test:", test_df["label"].nunique())

print("\nSTEP 3 COMPLETED SUCCESSFULLY")


Training images: 35149
Validation images: 17572
Test images: 7030

Classes in each split:
Train: 38
Validation: 38
Test: 38

STEP 3 COMPLETED SUCCESSFULLY


In [4]:

# STEP 4: Verify DenseNet121 preprocessing

import numpy as np
import tensorflow as tf

preprocess_input = tf.keras.applications.densenet.preprocess_input

# Test pixel values: black, middle gray, and white
sample_pixels = np.array(
    [[[[0, 0, 0],
       [127.5, 127.5, 127.5],
       [255, 255, 255]]]],
    dtype=np.float32
)

processed_pixels = preprocess_input(sample_pixels.copy())

print("Original pixel values:")
print(sample_pixels[0, 0])

print("\nProcessed pixel values:")
print(processed_pixels[0, 0])

print("\nMinimum processed value:", processed_pixels.min())
print("Maximum processed value:", processed_pixels.max())

# Check the expected DenseNet121 preprocessing behavior
assert np.isclose(processed_pixels[0, 0, 0, 0], -1.0)
assert np.isclose(processed_pixels[0, 0, 1, 0], 0.0)
assert np.isclose(processed_pixels[0, 0, 2, 0], 1.0)

print("\nSTEP 4 COMPLETED SUCCESSFULLY")


Original pixel values:
[[  0.    0.    0. ]
 [127.5 127.5 127.5]
 [255.  255.  255. ]]

Processed pixel values:
[[-2.117904   -2.0357141  -1.8044444 ]
 [ 0.06550212  0.19642857  0.41777784]
 [ 2.2489083   2.4285715   2.64      ]]

Minimum processed value: -2.117904
Maximum processed value: 2.64


AssertionError: 

In [ ]:

# STEP 4 (FIXED): Verify DenseNet121 preprocessing

import numpy as np
import tensorflow as tf

preprocess_input = tf.keras.applications.densenet.preprocess_input

# Test black, middle gray, and white pixels
sample_pixels = np.array(
    [[[[0, 0, 0],
       [127.5, 127.5, 127.5],
       [255, 255, 255]]]],
    dtype=np.float32
)

processed_pixels = preprocess_input(sample_pixels.copy())

print("Original pixel values:")
print(sample_pixels[0, 0])

print("\nDenseNet121 processed pixel values:")
print(processed_pixels[0, 0])

print("\nMinimum:", processed_pixels.min())
print("Maximum:", processed_pixels.max())

# Verify against the expected ImageNet channel-wise normalization
mean = np.array([0.485, 0.456, 0.406], dtype=np.float32)
std = np.array([0.229, 0.224, 0.225], dtype=np.float32)

expected = (sample_pixels / 255.0 - mean) / std

np.testing.assert_allclose(
    processed_pixels,
    expected,
    rtol=1e-5,
    atol=1e-5
)

print("\nSTEP 4 COMPLETED SUCCESSFULLY")


In [ ]:

# STEP 5: Create image-loading pipelines for DenseNet121

import numpy as np
import tensorflow as tf

IMAGE_SIZE = (224, 224)
BATCH_SIZE = 32

# Keep one consistent class-to-index mapping
class_names = sorted(class_names)
class_to_index = {
    name: index for index, name in enumerate(class_names)
}

def load_image(image_path, label):
    # Read image
    image = tf.io.read_file(image_path)

    # Decode as RGB
    image = tf.io.decode_image(
        image,
        channels=3,
        expand_animations=False
    )
    image.set_shape([None, None, 3])

    # Resize and convert pixels to float32 [0, 255]
    image = tf.image.resize(image, IMAGE_SIZE)
    image = tf.cast(image, tf.float32)

    # Apply DenseNet121 ImageNet preprocessing ONCE
    image = tf.keras.applications.densenet.preprocess_input(image)

    return image, label


def create_dataset(dataframe, training=False):
    paths = dataframe["image_path"].to_numpy()
    labels = dataframe["label"].map(class_to_index).to_numpy(
        dtype=np.int32
    )

    dataset = tf.data.Dataset.from_tensor_slices((paths, labels))

    if training:
        dataset = dataset.shuffle(
            buffer_size=min(len(dataframe), 10000),
            seed=42,
            reshuffle_each_iteration=True
        )

    dataset = dataset.map(
        load_image,
        num_parallel_calls=tf.data.AUTOTUNE
    )

    dataset = dataset.batch(BATCH_SIZE)
    dataset = dataset.prefetch(tf.data.AUTOTUNE)

    return dataset


train_ds = create_dataset(train_df, training=True)
valid_ds = create_dataset(valid_df)
test_ds = create_dataset(test_df)

# Inspect one training batch
for images, labels in train_ds.take(1):
    print("Image batch shape:", images.shape)
    print("Label batch shape:", labels.shape)
    print("Image data type:", images.dtype)
    print("Minimum pixel value:", float(tf.reduce_min(images)))
    print("Maximum pixel value:", float(tf.reduce_max(images)))
    print("Minimum label:", int(tf.reduce_min(labels)))
    print("Maximum label:", int(tf.reduce_max(labels)))

    assert images.shape[1:] == (224, 224, 3)
    assert images.shape[0] <= BATCH_SIZE
    assert images.dtype == tf.float32
    assert int(tf.reduce_min(labels)) >= 0
    assert int(tf.reduce_max(labels)) < len(class_names)

print("\nTraining batches:", len(train_ds))
print("Validation batches:", len(valid_ds))
print("Test batches:", len(test_ds))

print("\nSTEP 5 COMPLETED SUCCESSFULLY")


In [ ]:

# STEP 6: Build the DenseNet121 model

import tensorflow as tf

NUM_CLASSES = len(class_names)

# Load pretrained DenseNet121
base_model = tf.keras.applications.DenseNet121(
    weights="imagenet",
    include_top=False,
    input_shape=(224, 224, 3)
)

# Freeze pretrained feature-extraction layers for Stage 1
base_model.trainable = False

# Build classification model
inputs = tf.keras.Input(shape=(224, 224, 3), name="input_image")

# Images are already preprocessed by train_ds/valid_ds/test_ds
x = base_model(inputs, training=False)
x = tf.keras.layers.GlobalAveragePooling2D()(x)
x = tf.keras.layers.Dropout(0.3)(x)

outputs = tf.keras.layers.Dense(
    NUM_CLASSES,
    activation="softmax",
    name="disease_predictions"
)(x)

model = tf.keras.Model(
    inputs=inputs,
    outputs=outputs,
    name="Smart_Crop_Care_DenseNet121"
)

# Compile the model
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

# Display model information
print("Model name:", model.name)
print("Number of classes:", NUM_CLASSES)
print("Total parameters:", model.count_params())
print(
    "Trainable parameters:",
    sum(
        int(tf.size(variable).numpy())
        for variable in model.trainable_variables
    )
)
print(
    "Non-trainable parameters:",
    sum(
        int(tf.size(variable).numpy())
        for variable in model.non_trainable_variables
    )
)

print("\nModel output shape:", model.output_shape)
print("Base model trainable:", base_model.trainable)

assert NUM_CLASSES == 38
assert model.output_shape == (None, 38)
assert base_model.trainable is False

print("\nSTEP 6 COMPLETED SUCCESSFULLY")

# Uncomment this line if you want to inspect the full architecture:
# model.summary()


In [ ]:

# STEP 7: Configure Stage 1 training callbacks

from pathlib import Path
import tensorflow as tf

# Save model checkpoints in Kaggle's writable output directory
OUTPUT_DIR = Path("/kaggle/working/smart_crop_care")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

STAGE1_CHECKPOINT = OUTPUT_DIR / "best_densenet121_stage1.keras"

stage1_callbacks = [
    tf.keras.callbacks.ModelCheckpoint(
        filepath=str(STAGE1_CHECKPOINT),
        monitor="val_accuracy",
        mode="max",
        save_best_only=True,
        verbose=1
    ),

    tf.keras.callbacks.EarlyStopping(
        monitor="val_loss",
        patience=3,
        restore_best_weights=True,
        verbose=1
    ),

    tf.keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.2,
        patience=2,
        min_lr=1e-6,
        verbose=1
    )
]

STAGE1_EPOCHS = 10

print("Output directory:", OUTPUT_DIR)
print("Checkpoint path:", STAGE1_CHECKPOINT)
print("Maximum Stage 1 epochs:", STAGE1_EPOCHS)
print("Callbacks configured:", len(stage1_callbacks))
print("DenseNet121 base trainable:", base_model.trainable)

assert base_model.trainable is False
assert len(stage1_callbacks) == 3

print("\nSTEP 7 COMPLETED SUCCESSFULLY")


In [ ]:

# STEP 8: Train DenseNet121 - Stage 1

history_stage1 = model.fit(
    train_ds,
    validation_data=valid_ds,
    epochs=STAGE1_EPOCHS,
    callbacks=stage1_callbacks,
    verbose=1
)

print("\nSTAGE 1 TRAINING COMPLETED")

print(
    "Best validation accuracy:",
    round(max(history_stage1.history["val_accuracy"]), 4)
)

print(
    "Best validation loss:",
    round(min(history_stage1.history["val_loss"]), 4)
)

print("Checkpoint exists:", STAGE1_CHECKPOINT.exists())
print("Checkpoint path:", STAGE1_CHECKPOINT)


In [ ]:

# STEP 9: Prepare DenseNet121 for Stage 2 fine-tuning

import tensorflow as tf

# Load the best Stage 1 checkpoint
model = tf.keras.models.load_model(
    str(STAGE1_CHECKPOINT)
)

# Find the DenseNet121 feature extractor
base_model = model.get_layer("densenet121")

# Unfreeze the base model, then freeze most layers
base_model.trainable = True

# Fine-tune only the final 40 layers
for layer in base_model.layers[:-40]:
    layer.trainable = False

for layer in base_model.layers[-40:]:
    # Keep BatchNormalization frozen for stable fine-tuning
    if isinstance(layer, tf.keras.layers.BatchNormalization):
        layer.trainable = False
    else:
        layer.trainable = True

# Recompile with a small learning rate
model.compile(
    optimizer=tf.keras.optimizers.Adam(
        learning_rate=1e-5
    ),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

print("Loaded Stage 1 checkpoint successfully")
print("Model output shape:", model.output_shape)
print("Base model trainable:", base_model.trainable)
print(
    "Trainable parameters:",
    sum(
        int(tf.size(v).numpy())
        for v in model.trainable_variables
    )
)
print(
    "Non-trainable parameters:",
    sum(
        int(tf.size(v).numpy())
        for v in model.non_trainable_variables
    )
)
print("Fine-tuning learning rate: 0.00001")

assert model.output_shape == (None, 38)
assert base_model.trainable is True

print("\nSTEP 9 COMPLETED SUCCESSFULLY")


In [ ]:

# STEP 10: Configure Stage 2 fine-tuning callbacks

STAGE2_CHECKPOINT = OUTPUT_DIR / "best_densenet121_stage2.keras"

stage2_callbacks = [
    tf.keras.callbacks.ModelCheckpoint(
        filepath=str(STAGE2_CHECKPOINT),
        monitor="val_accuracy",
        mode="max",
        save_best_only=True,
        verbose=1
    ),

    tf.keras.callbacks.EarlyStopping(
        monitor="val_loss",
        patience=3,
        restore_best_weights=True,
        verbose=1
    ),

    tf.keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.2,
        patience=2,
        min_lr=1e-7,
        verbose=1
    )
]

STAGE2_EPOCHS = 8

print("Stage 2 checkpoint:", STAGE2_CHECKPOINT)
print("Maximum Stage 2 epochs:", STAGE2_EPOCHS)
print("Callbacks configured:", len(stage2_callbacks))
print("Current learning rate:",
      float(tf.keras.backend.get_value(model.optimizer.learning_rate)))

assert len(stage2_callbacks) == 3
assert model.output_shape == (None, 38)

print("\nSTEP 10 COMPLETED SUCCESSFULLY")


In [ ]:

# STEP 11: Fine-tune DenseNet121 - Stage 2

history_stage2 = model.fit(
    train_ds,
    validation_data=valid_ds,
    epochs=STAGE2_EPOCHS,
    callbacks=stage2_callbacks,
    verbose=1
)

print("\nSTAGE 2 TRAINING COMPLETED")

print(
    "Best validation accuracy:",
    round(max(history_stage2.history["val_accuracy"]), 4)
)

print(
    "Best validation loss:",
    round(min(history_stage2.history["val_loss"]), 4)
)

print("Checkpoint exists:", STAGE2_CHECKPOINT.exists())
print("Checkpoint path:", STAGE2_CHECKPOINT)


In [ ]:

# STEP 12: Load and verify the best Stage 2 model

import tensorflow as tf
from pathlib import Path

assert STAGE2_CHECKPOINT.exists(), (
    f"Checkpoint not found: {STAGE2_CHECKPOINT}"
)

best_model = tf.keras.models.load_model(
    str(STAGE2_CHECKPOINT)
)

print("Loaded model:", best_model.name)
print("Model output shape:", best_model.output_shape)
print("Number of classes:", len(class_names))
print("Checkpoint size (MB):", round(
    STAGE2_CHECKPOINT.stat().st_size / (1024 * 1024), 2
))

# Run one test batch to verify inference
for images, labels in test_ds.take(1):
    predictions = best_model.predict(images, verbose=0)

    print("\nInput batch shape:", images.shape)
    print("Prediction shape:", predictions.shape)
    print("First actual label:", int(labels[0]))
    print("First predicted label:", int(predictions[0].argmax()))
    print("First prediction confidence:", round(
        float(predictions[0].max()), 4
    ))

    assert predictions.shape == (images.shape[0], 38)
    assert np.allclose(predictions.sum(axis=1), 1.0, atol=1e-4)

print("\nSTEP 12 COMPLETED SUCCESSFULLY")


In [ ]:

# STEP 13: Final evaluation on the held-out test set

import numpy as np
import tensorflow as tf
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix
)

# Evaluate loss and accuracy
test_loss, test_accuracy = best_model.evaluate(
    test_ds,
    verbose=1
)

# Collect predictions and true labels
y_true = []
y_pred = []

for images, labels in test_ds:
    probabilities = best_model.predict(images, verbose=0)

    y_true.extend(labels.numpy().tolist())
    y_pred.extend(np.argmax(probabilities, axis=1).tolist())

y_true = np.array(y_true)
y_pred = np.array(y_pred)

final_accuracy = accuracy_score(y_true, y_pred)

print("\n========== FINAL TEST RESULTS ==========")
print(f"Test images: {len(y_true)}")
print(f"Test loss: {test_loss:.4f}")
print(f"Test accuracy: {test_accuracy * 100:.2f}%")
print(f"Calculated accuracy: {final_accuracy * 100:.2f}%")

print("\n========== CLASSIFICATION REPORT ==========")
print(classification_report(
    y_true,
    y_pred,
    labels=list(range(len(class_names))),
    target_names=class_names,
    digits=4,
    zero_division=0
))

# Create confusion matrix for later analysis
cm = confusion_matrix(
    y_true,
    y_pred,
    labels=list(range(len(class_names)))
)

print("Confusion matrix shape:", cm.shape)
print("Total test predictions:", cm.sum())

assert len(y_true) == len(test_df)
assert cm.shape == (38, 38)
assert cm.sum() == len(test_df)

print("\nSTEP 13 COMPLETED SUCCESSFULLY")


In [ ]:

# STEP 14: Save final model and class labels

import json
from pathlib import Path
import tensorflow as tf

FINAL_DIR = Path("/kaggle/working/smart_crop_care/final")
FINAL_DIR.mkdir(parents=True, exist_ok=True)

FINAL_MODEL_PATH = FINAL_DIR / "smart_crop_care_densenet121.keras"
CLASS_NAMES_PATH = FINAL_DIR / "class_names.json"

# Load the best Stage 2 checkpoint and save the final model
final_model = tf.keras.models.load_model(
    str(STAGE2_CHECKPOINT)
)

final_model.save(str(FINAL_MODEL_PATH))

# Save class names in the exact numerical label order
with open(CLASS_NAMES_PATH, "w", encoding="utf-8") as f:
    json.dump(class_names, f, indent=2, ensure_ascii=False)

# Verify saved files
assert FINAL_MODEL_PATH.exists()
assert CLASS_NAMES_PATH.exists()

with open(CLASS_NAMES_PATH, "r", encoding="utf-8") as f:
    saved_class_names = json.load(f)

assert saved_class_names == class_names
assert final_model.output_shape == (None, 38)

print("Final model saved:", FINAL_MODEL_PATH)
print("Model size (MB):", round(
    FINAL_MODEL_PATH.stat().st_size / (1024 * 1024), 2
))
print("Class names saved:", CLASS_NAMES_PATH)
print("Number of saved classes:", len(saved_class_names))

print("\nSTEP 14 COMPLETED SUCCESSFULLY")


In [ ]:

# STEP 15: Package final model files for download

import zipfile
from pathlib import Path

FINAL_DIR = Path("/kaggle/working/smart_crop_care/final")
ZIP_PATH = Path("/kaggle/working/smart_crop_care_densenet121.zip")

MODEL_PATH = FINAL_DIR / "smart_crop_care_densenet121.keras"
LABELS_PATH = FINAL_DIR / "class_names.json"

assert MODEL_PATH.exists(), "Model file is missing"
assert LABELS_PATH.exists(), "Class names file is missing"

with zipfile.ZipFile(
    ZIP_PATH, mode="w", compression=zipfile.ZIP_DEFLATED
) as zip_file:
    zip_file.write(MODEL_PATH, arcname=MODEL_PATH.name)
    zip_file.write(LABELS_PATH, arcname=LABELS_PATH.name)

# Verify ZIP contents
with zipfile.ZipFile(ZIP_PATH, "r") as zip_file:
    contents = zip_file.namelist()
    assert MODEL_PATH.name in contents
    assert LABELS_PATH.name in contents
    assert zip_file.testzip() is None

print("ZIP file:", ZIP_PATH)
print("ZIP size (MB):", round(ZIP_PATH.stat().st_size / (1024 * 1024), 2))
print("Files inside ZIP:")
for name in contents:
    print("-", name)

print("\nSTEP 15 COMPLETED SUCCESSFULLY")
